# Notebook 05 — Exploratory Churn Analysis (EDA) & Visual Asset Generation
## Telecom Customer Churn and Retention Intelligence Dashboard

---

**Purpose:** Conduct comprehensive statistical exploration of churn drivers, examine bivariate and multivariate relationships, compute revenue impacts, and export high-resolution chart assets for executive presentations and report documentation.

**Input:** `data/processed/telco_segmented.csv`

**Visual Assets Generated:**
1. `assets/churn_by_contract.png` — Contract type churn comparison
2. `assets/churn_by_tenure_cohort.png` — Lifecycle cohort attrition rate
3. `assets/churn_by_internet_techsupport.png` — The "Security & Support Shield" effect
4. `assets/churn_by_payment_method.png` — Payment method friction analysis
5. `assets/correlation_matrix.png` — Numerical correlation heatmap

---

## Section 1 — Setup, Matplotlib Styling & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

# Set consistent corporate aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.titlesize'] = 16

PRIMARY_COLOR = '#1E3A8A'   # Corporate Navy
CHURN_COLOR = '#DC2626'     # Alert Red
RETAIN_COLOR = '#059669'    # Emerald Green
ACCENT_COLOR = '#F59E0B'    # Amber

PROJECT_ROOT = Path.cwd().parent
DATA_FILE = PROJECT_ROOT / 'data' / 'processed' / 'telco_segmented.csv'
ASSETS_DIR = PROJECT_ROOT / 'assets'
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_FILE)
print(f'Segmented dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

## Section 2 — Baseline Financial Overview & Churn Impact

In [ ]:
total_customers = len(df)
total_churned = df['churn_flag'].sum()
churn_rate = (total_churned / total_customers) * 100
total_mrr = df['monthly_charges'].sum()
churned_mrr = df[df['churn_flag'] == 1]['monthly_charges'].sum()
mrr_churn_pct = (churned_mrr / total_mrr) * 100
annualized_loss = churned_mrr * 12

print('='*55)
print('TELECOM PORTFOLIO EXECUTIVE FINANCIAL SUMMARY')
print('='*55)
print(f'Total Customer Base:         {total_customers:,}')
print(f'Total Churned Customers:     {total_churned:,} ({churn_rate:.2f}%)')
print(f'Total Monthly Recurring Rev: ${total_mrr:,.2f}')
print(f'Monthly Revenue Lost (MRR):  ${churned_mrr:,.2f} ({mrr_churn_pct:.2f}%)')
print(f'Annualized Lost Revenue:     ${annualized_loss:,.2f}')
print('='*55)

## Section 3 — Churn Driver #1: Contract Commitment

**Finding:** Month-to-month contracts have a staggering **42.7%** churn rate compared to **11.3%** for One-year and **2.8%** for Two-year contracts.

In [ ]:
contract_df = df.groupby('contract').agg(
    total=('customer_id', 'count'),
    churned=('churn_flag', 'sum'),
    churn_rate=('churn_flag', lambda x: x.mean() * 100)
).reindex(['Month-to-month', 'One year', 'Two year'])

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(contract_df.index, contract_df['churn_rate'], color=[CHURN_COLOR, ACCENT_COLOR, RETAIN_COLOR], width=0.55, edgecolor='black', alpha=0.9)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 1.2, f'{yval:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_title('Churn Rate by Contract Type (Commitment Factor)', pad=15)
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 50)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()

output_chart = ASSETS_DIR / 'churn_by_contract.png'
plt.savefig(output_chart, dpi=300)
plt.show()
print(f'Saved: {output_chart}')

## Section 4 — Churn Driver #2: Tenure Lifecycle (First-Year Cliff)

**Finding:** Almost half of first-year customers churn within the first 12 months (**47.4%** churn rate), whereas customers who stay past Year 2 drop to under **15%**.

In [ ]:
cohort_df = df.groupby('tenure_cohort', observed=False).agg(
    total=('customer_id', 'count'),
    churned=('churn_flag', 'sum'),
    churn_rate=('churn_flag', lambda x: x.mean() * 100)
).reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(cohort_df['tenure_cohort'], cohort_df['churn_rate'], color=[CHURN_COLOR, '#EA580C', '#0284C7', RETAIN_COLOR], width=0.55, edgecolor='black', alpha=0.9)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 1.2, f'{yval:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_title('Churn Rate Across Customer Lifecycle (Tenure Cohorts)', pad=15)
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 55)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()

output_chart = ASSETS_DIR / 'churn_by_tenure_cohort.png'
plt.savefig(output_chart, dpi=300)
plt.show()
print(f'Saved: {output_chart}')

## Section 5 — Churn Driver #3: The "Protection & Support Shield"

**Finding:** Fiber Optic customers have high churn (41.9%), but having **Tech Support** and **Online Security** reduces churn by more than half!

In [ ]:
shield_df = df[df['internet_service'] != 'No'].groupby(['tech_support', 'online_security']).agg(
    total=('customer_id', 'count'),
    churn_rate=('churn_flag', lambda x: x.mean() * 100)
).reset_index()

shield_df['Protection Category'] = 'TechSupport: ' + shield_df['tech_support'] + ' | Security: ' + shield_df['online_security']
shield_df = shield_df.sort_values(by='churn_rate', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(shield_df['Protection Category'], shield_df['churn_rate'], color=[CHURN_COLOR, '#EA580C', '#0284C7', RETAIN_COLOR], edgecolor='black', alpha=0.9, height=0.55)

for bar in bars:
    xval = bar.get_width()
    ax.text(xval + 0.8, bar.get_y() + bar.get_height()/2, f'{xval:.1f}%', ha='left', va='center', fontweight='bold', fontsize=11)

ax.set_title('The Protection Shield: Churn Rate by Tech Support & Security', pad=15)
ax.set_xlabel('Churn Rate (%)')
ax.set_xlim(0, 55)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.invert_yaxis()
plt.tight_layout()

output_chart = ASSETS_DIR / 'churn_by_internet_techsupport.png'
plt.savefig(output_chart, dpi=300)
plt.show()
print(f'Saved: {output_chart}')

## Section 6 — Churn Driver #4: Payment Method Friction

**Finding:** Electronic check users churn at **45.3%**, whereas automated bank transfer/credit card customers churn at only **16.0%**.

In [ ]:
pay_df = df.groupby('payment_method').agg(
    total=('customer_id', 'count'),
    churn_rate=('churn_flag', lambda x: x.mean() * 100)
).sort_values('churn_rate', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(pay_df.index, pay_df['churn_rate'], color=[CHURN_COLOR, '#EA580C', RETAIN_COLOR, RETAIN_COLOR], width=0.55, edgecolor='black', alpha=0.9)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 1.2, f'{yval:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('Churn Rate by Payment Method (Friction Factor)', pad=15)
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 55)
plt.xticks(rotation=15, ha='right')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()

output_chart = ASSETS_DIR / 'churn_by_payment_method.png'
plt.savefig(output_chart, dpi=300)
plt.show()
print(f'Saved: {output_chart}')

## Section 7 — Numerical Correlation Heatmap

**Finding:** Risk score (`0.58`) and tenure (`-0.35`) show the strongest positive and negative linear correlations with customer churn.

In [ ]:
numeric_cols = [
    'churn_flag', 'risk_score', 'tenure', 'monthly_charges', 'total_charges',
    'total_services_subscribed', 'security_tech_score', 'auto_payment_flag', 'high_risk_contract'
]

corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
cax = ax.matshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
fig.colorbar(cax, fraction=0.046, pad=0.04)

ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha='left', fontsize=10)
ax.set_yticklabels(numeric_cols, fontsize=10)

# Annotate numbers
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        val = corr_matrix.iloc[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                color='white' if abs(val) > 0.5 else 'black', fontsize=9, fontweight='bold')

ax.set_title('Correlation Heatmap — Churn Drivers & Behavioral Features', pad=30)
plt.tight_layout()

output_chart = ASSETS_DIR / 'correlation_matrix.png'
plt.savefig(output_chart, dpi=300)
plt.show()
print(f'Saved: {output_chart}')

## Section 8 — Key Executive Takeaways Summary Table

In [ ]:
eda_takeaways = pd.DataFrame({
    'Churn Driver': [
        'Contract Type',
        'Tenure Lifecycle',
        'Payment Method',
        'Support Services',
        'Network Technology'
    ],
    'High-Risk Group': [
        'Month-to-month (42.7% Churn)',
        '0-12 Months Tenure (47.4% Churn)',
        'Electronic Check (45.3% Churn)',
        'No Tech Support / Security (41.7% Churn)',
        'Fiber Optic (41.9% Churn)'
    ],
    'Low-Risk Group': [
        'Two-Year Contract (2.8% Churn)',
        '49-72 Months Tenure (6.6% Churn)',
        'Bank/Card Autopay (16.0% Churn)',
        'With Tech Support + Security (14.6% Churn)',
        'No Internet / Phone Only (7.4% Churn)'
    ],
    'Strategic Action Recommendation': [
        'Incentivize 1-Yr contract migration with discounts',
        'Proactive 30/60/90 day customer onboarding concierge',
        'Offer $5/mo bill credit for switching to Autopay',
        'Free 90-day Tech Support bundle for new Fiber accounts',
        'Service quality audit & price transparency communication'
    ]
})

display(eda_takeaways)

---

## Next Step

Proceed to **Phase 8 — SQL Database Pipeline** (`sql/` scripts) where we:
1. Define DDL database schemas (`schema.sql`)
2. Load processed data into SQLite / PostgreSQL / BigQuery
3. Build analytical views and executive KPI queries

---
**Exploratory Churn Analysis Phase Complete.**